# YOLOv11 + DeepSORT Tracking Pipeline

Written by Adit, reran by Evan on Mac w/ GPU and CPU

In [ ]:
!pip install ultralytics torch opencv-python deep-sort-realtime tqdm numpy

In [1]:
import os
import cv2
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
import time
import json
import csv
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.9.1


In [2]:
# Detect if running on Google Colab
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    REPO_ROOT = Path("/content/sports-data-tracker")

    # if running notebook from within the notebooks folder
    if not REPO_ROOT.exists():
        REPO_ROOT = Path("/content/drive/MyDrive/sports-data-tracker")  # using Google Drive
    if not REPO_ROOT.exists():
        REPO_ROOT = Path(os.getcwd()).parent.resolve()  # fallback
else:
    # local
    REPO_ROOT = Path(os.getcwd()).parent.resolve()

DATA_ROOT = REPO_ROOT / "data" / "soccer_side"
RESULTS_ROOT = REPO_ROOT / "results" / "yolov11_deepsort"

# results directories
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
(RESULTS_ROOT / "mot_outputs").mkdir(exist_ok=True)
(RESULTS_ROOT / "videos").mkdir(exist_ok=True)

print(f"Running on Colab: {IN_COLAB}")
print(f"Repo root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Results root: {RESULTS_ROOT}")

# verify data
if DATA_ROOT.exists():
    test_dir = DATA_ROOT / "test"
    if test_dir.exists():
        num_sequences = len([d for d in test_dir.iterdir() if d.is_dir()])
        print(f"Found {num_sequences} test sequences in {test_dir}")
    else:
        print(f"Test directory not found: {test_dir}")
else:
    print(f"Data root not found: {DATA_ROOT}")

# Check for Apple Silicon (MPS) or NVIDIA (CUDA)
if torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple MPS (Metal Performance Shaders) acceleration")
elif torch.cuda.is_available():
    DEVICE = "cuda"
    print("Using NVIDIA CUDA acceleration")
else:
    DEVICE = "cpu"
    print("Using CPU")

print(f"Using device: {DEVICE}")

# detection / tracking configs (copied from yolov11_botsort.ipynb)
CONF_THRESHOLD = 0.25  # bumped from 0.15 to cut false positives
IOU_THRESHOLD = 0.5
PERSON_CLASS_ID = 0

# IMPORTANT: wide soccer videos need more resolution for small players
IMGSZ = 1920

# DeepSORT configs
MAX_AGE = 25
N_INIT = 3
MAX_IOU_DISTANCE = 0.7
MAX_COSINE_DISTANCE = 0.3

# video configs
VIDEO_CODEC = 'mp4v'
VIDEO_EXTS = {".mp4", ".mkv", ".avi", ".mov"}

# viz configs
BBOX_THICKNESS = 2
TEXT_FONT = cv2.FONT_HERSHEY_SIMPLEX
TEXT_SCALE = 0.6
TEXT_THICKNESS = 2
TEXT_Y_OFFSET = 10

Running on Colab: False
Repo root: /Users/echen/Documents/FA25/543/sports-data-tracker
Data root: /Users/echen/Documents/FA25/543/sports-data-tracker/data/soccer_side
Results root: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort
Found 10 test sequences in /Users/echen/Documents/FA25/543/sports-data-tracker/data/soccer_side/test
Using Apple MPS (Metal Performance Shaders) acceleration
Using device: mps


In [3]:
class YOLOv11Detector:
    YOLO_WEIGHTS = "yolo11n.pt"

    def __init__(self, conf_threshold, iou_threshold, imgsz, device):
        """
        Initialize YOLOv11 detector.

        Parameters:
        -----------
        conf_threshold : float
            Minimum confidence score for detections (0-1)
        iou_threshold : float
            IoU threshold for non-maximum suppression
        imgsz : int
            Inference image size (larger helps with small objects)
        device : str
            "cuda" or "cpu"
        """
        self.conf_threshold = conf_threshold
        self.iou_threshold = iou_threshold
        self.imgsz = imgsz
        self.device = device

        print(f"Loading YOLOv11 weights: {self.YOLO_WEIGHTS}")
        self.model = YOLO(self.YOLO_WEIGHTS)
        self.model.to(device)
        print(f"YOLOv11 model loaded on {device}")
        print(f"imgsz set to {imgsz} for small player detection")

    def detect(self, frame, filter_classes=None):
        """
        Detect objects in a single frame.

        Parameters:
        -----------
        frame : np.ndarray
            BGR image from OpenCV (ultrawide: 6500x1000)
        filter_classes : list or None
            List of class IDs to keep (e.g., [0] for person only)

        Returns:
        --------
        detections : list of tuples
            Each tuple: ([x, y, w, h], confidence, class_id)
            Bounding box in [left, top, width, height] format
        """
        # run YOLOv11 detection 
        results = self.model(
            frame,
            conf=self.conf_threshold,
            iou=self.iou_threshold,
            imgsz=self.imgsz,
            device=self.device,
            verbose=False
        )

        detections = []
        if results and len(results) > 0:
            result = results[0]
            boxes = result.boxes

            if boxes is not None and len(boxes) > 0:
                xyxy = boxes.xyxy.cpu().numpy()
                scores = boxes.conf.cpu().numpy()
                classes = boxes.cls.cpu().numpy().astype(int)

                for (x1, y1, x2, y2), score, class_id in zip(xyxy, scores, classes):
                    if filter_classes is not None and class_id not in filter_classes:
                        continue

                    # [x1, y1, x2, y2] to [x, y, w, h]
                    x = int(x1)
                    y = int(y1)
                    w = int(x2 - x1)
                    h = int(y2 - y1)

                    # valid bounding box
                    x = max(0, x)
                    y = max(0, y)
                    w = max(1, w)
                    h = max(1, h)

                    detections.append(([x, y, w, h], float(score), class_id))

        return detections

In [4]:
class DeepSORTTracker:
    """
    DeepSORT tracker wrapper for multi-object tracking: motion prediction with appearance features for robust tracking
    """

    def __init__(
        self,
        max_age=30,
        n_init=3,
        max_iou_distance=0.7,
        max_cosine_distance=0.3,
        embedder="mobilenet",
        embedder_gpu=True,
    ):
        """
        Initialize DeepSORT tracker

        Parameters:
        -----------
        max_age : int
            Maximum frames to keep a track alive without associated detections
        n_init : int
            Number of consecutive detections before track is confirmed
        max_iou_distance : float
            Maximum IoU distance for matching (lower = stricter)
        max_cosine_distance : float
            Maximum cosine distance for appearance matching (lower = stricter)
        embedder : str
            Name of the embedder model for appearance features
        embedder_gpu : bool
            Whether to use GPU for embedder
        """
        self.tracker = DeepSort(
            max_age=max_age,
            n_init=n_init,
            max_iou_distance=max_iou_distance,
            max_cosine_distance=max_cosine_distance,
            embedder=embedder,
            embedder_gpu=embedder_gpu
        )

        print(f"DeepSORT tracker initialized for ultrawide frames:")
        print(f"  - max_age: {max_age}")
        print(f"  - n_init: {n_init}")
        print(f"  - max_iou_distance: {max_iou_distance}")
        print(f"  - max_cosine_distance: {max_cosine_distance}")
        print(f"  - embedder: {embedder}")

    def update(self, detections, frame):
        """
        Update tracker with new detections

        Parameters:
        -----------
        detections : list of tuples
            Each tuple: ([x, y, w, h], confidence, class_id)
        frame : np.ndarray
            Current frame (BGR, ultrawide 6500x1000) for appearance feature extraction

        Returns:
        --------
        tracks : list of tuples
            Each tuple: (track_id, [x, y, w, h], confidence)
        """
        if len(detections) == 0:
            # update empty detections to age out tracks
            self.tracker.update_tracks([], frame=frame)
            return []

        # convert to DeepSORT format: ([left, top, w, h], confidence, class)
        deepsort_detections = []
        for bbox, conf, class_id in detections:
            deepsort_detections.append((bbox, conf, class_id))

        # update tracker
        tracks = self.tracker.update_tracks(deepsort_detections, frame=frame)

        # extract confirmed tracks
        results = []
        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = track.track_id
            if isinstance(track_id, str):
                track_id = int(track_id) if track_id.isdigit() else hash(track_id) % 10000
            else:
                track_id = int(track_id)

            ltrb = track.to_ltrb() # [left, top, right, bottom]

            # convert to [x, y, w, h]
            x = int(ltrb[0])
            y = int(ltrb[1])
            w = int(ltrb[2] - ltrb[0])
            h = int(ltrb[3] - ltrb[1])

            # get detection confidence
            conf = track.det_conf if track.det_conf is not None else 1.0

            results.append((track_id, [x, y, w, h], conf))

        return results

    def reset(self):
        """Reset tracker state for new sequence."""
        self.tracker.delete_all_tracks()

In [5]:
def get_color_for_id(track_id):
    """
    Generate a color for a track ID - deterministic based on track_id
    """
    rng = np.random.RandomState(int(track_id))
    return tuple(int(c) for c in rng.randint(0, 255, 3))


def draw_tracks(frame, tracks):
    """
    Draw bounding boxes and track IDs on frame

    Parameters:
    -----------
    frame : np.ndarray
        BGR image to draw on
    tracks : list of tuples
        Each tuple: (track_id, [x, y, w, h], confidence)

    Returns:
    --------
    frame : np.ndarray
        Annotated frame
    """
    for track_id, bbox, conf in tracks:
        x, y, w, h = bbox
        color = get_color_for_id(track_id)

        # bounding box
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, BBOX_THICKNESS)

        # track ID and confidence
        label = f"ID:{track_id} ({conf:.2f})"
        cv2.putText(frame, label, (x, y - TEXT_Y_OFFSET),
                    TEXT_FONT, TEXT_SCALE, color, TEXT_THICKNESS)

    return frame

In [6]:
def discover_sequences(data_root: Path):
    """
    Discover all sequences under data_root that:
    - Have a gt/gt.txt file
    - Have at least one video file in the same parent directory

    Prefers _fixed.mp4 files over regular .mp4 files
    """
    seqs = []

    for gt_file in data_root.rglob("gt.txt"):
        gt_dir = gt_file.parent
        seq_dir = gt_dir.parent
        video_files = [
            p for p in seq_dir.iterdir()
            if p.is_file() and p.suffix.lower() in VIDEO_EXTS
        ]

        if not video_files:
            continue

        # Prefer _fixed.mp4 files over regular .mp4 files
        fixed_videos = [v for v in video_files if "_fixed.mp4" in v.name]
        if fixed_videos:
            video_files = fixed_videos
        else:
            mp4_videos = [v for v in video_files if v.suffix.lower() == ".mp4"]
            if mp4_videos:
                video_files = mp4_videos

        video_files.sort()
        video_path = video_files[0]

        if len(video_files) > 1:
            print(f"Warning: Multiple videos in {seq_dir}, using {video_path.name}")

        seqs.append({
            "name": seq_dir.name,
            "seq_dir": seq_dir,
            "gt_path": gt_file,
            "video_path": video_path,
        })

    return seqs

# discover sequences
sequences = discover_sequences(DATA_ROOT)

print(f"\nDiscovered {len(sequences)} sequence(s) with video + ground truth:")
for s in sequences:
    print(f"  - {s['name']}: {s['video_path'].name}")


Discovered 22 sequence(s) with video + ground truth:
  - F_20220220_1_1890_1920: img1.mp4
  - F_20220220_1_1920_1950: img1.mp4
  - F_20220220_1_1680_1710: img1.mp4
  - F_20220220_1_1770_1800: img1.mp4
  - F_20220220_1_1950_1980: img1.mp4
  - F_20220220_1_1830_1860: img1.mp4
  - F_20220220_1_1740_1770: img1.mp4
  - F_20220220_1_1860_1890: img1.mp4
  - F_20220220_1_1800_1830: img1.mp4
  - F_20220220_1_1710_1740: img1.mp4
  - F_20200220_1_0180_0210: img1.mp4
  - F_20220220_1_1080_1110: img1.mp4
  - F_20200220_1_0330_0360: img1.mp4
  - F_20200220_1_0060_0090: img1.mp4
  - F_20220220_1_1260_1290: img1.mp4
  - F_20220220_1_0960_0990: img1.mp4
  - F_20220220_1_1200_1230: img1.mp4
  - F_20220220_1_0900_0930: img1.mp4
  - F_20220220_1_0990_1020: img1.mp4
  - F_20200220_1_0780_0810: img1.mp4
  - F_20200220_1_0690_0720: img1.mp4
  - F_20200220_1_0540_0570: img1.mp4


## Main Processing Pipeline


In [7]:
def process_sequence(
    detector: YOLOv11Detector,
    tracker: DeepSORTTracker,
    video_path: Path,
    seq_name: str,
    output_dir: Path,
    filter_classes=(PERSON_CLASS_ID,),
    save_video=True
):
    """
    Process a single video sequence with YOLOv11 + DeepSORT

    Parameters:
    -----------
    detector : YOLOv11Detector
        YOLOv11 detector instance
    tracker : DeepSORTTracker
        DeepSORT tracker instance
    video_path : Path
        Path to input video file
    seq_name : str
        Name of the sequence
    output_dir : Path
        Directory to save results
    filter_classes : tuple
        Class IDs to detect (default: person only)
    save_video : bool
        Whether to save annotated video

    Returns:
    --------
    metrics : dict
        Processing metrics (time, FPS, frame count, etc.)
    """
    # reset tracker for new sequence
    tracker.reset()

    # open video
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return None

    # get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # setup output paths
    mot_output_dir = output_dir / "mot_outputs"
    mot_output_dir.mkdir(exist_ok=True)
    mot_file_path = mot_output_dir / f"{seq_name}.txt"
    video_output_dir = output_dir / "videos"
    video_output_dir.mkdir(exist_ok=True)
    video_output_path = video_output_dir / f"{seq_name}_tracked.mp4"

    # video writer
    video_writer = None
    if save_video:
        fourcc = cv2.VideoWriter_fourcc(*VIDEO_CODEC)
        video_writer = cv2.VideoWriter(str(video_output_path), fourcc, fps, (width, height))

    # MOT output file
    mot_file = open(mot_file_path, 'w', newline='')
    mot_writer = csv.writer(mot_file)

    # setup processing
    frame_idx = 0
    total_detections = 0
    start_time = time.time()

    progress_bar = tqdm(total=total_frames, desc=f"Processing {seq_name}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        # detect objects with YOLOv11
        detections = detector.detect(frame, filter_classes=filter_classes)

        # update tracker with detections
        tracks = tracker.update(detections, frame)

        # write MOT format results
        for track_id, bbox, conf in tracks:
            x, y, w, h = bbox
            mot_writer.writerow([
                frame_idx,
                int(track_id),
                float(x),
                float(y),
                float(w),
                float(h),
                float(conf),
                -1, -1, -1
            ])
            total_detections += 1

        # draw and save frame
        if save_video and video_writer is not None:
            annotated_frame = draw_tracks(frame.copy(), tracks)
            video_writer.write(annotated_frame)

        progress_bar.update(1)

    progress_bar.close()

    # cleanup
    cap.release()
    mot_file.close()
    if video_writer is not None:
        video_writer.release()

    # calculate metrics
    end_time = time.time()
    processing_time = end_time - start_time
    average_fps = frame_idx / processing_time if processing_time > 0 else 0

    print(f"[{seq_name}] Processed {frame_idx} frames in {processing_time:.2f}s ({average_fps:.2f} FPS)")
    print(f"[{seq_name}] Total tracks written: {total_detections}")
    print(f"[{seq_name}] MOT output: {mot_file_path}")
    if save_video:
        print(f"[{seq_name}] Video output: {video_output_path}")

    return {
        'sequence_name': seq_name,
        'total_frames': frame_idx,
        'total_detections': total_detections,
        'processing_time': processing_time,
        'average_fps': average_fps,
        'mot_file': str(mot_file_path),
        'video_file': str(video_output_path) if save_video else None
    }

## Initialize Detector and DeepSORT Models


In [8]:
detector = YOLOv11Detector(
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
    imgsz=IMGSZ,
    device=DEVICE 
)

# For DeepSORT force the embedder to mac CPU to avoid library conflicts
embedder_gpu = True if DEVICE == "cuda" else False

tracker = DeepSORTTracker(
    max_age=MAX_AGE,
    n_init=N_INIT,
    max_iou_distance=MAX_IOU_DISTANCE,
    max_cosine_distance=MAX_COSINE_DISTANCE,
    embedder="mobilenet",
    embedder_gpu=embedder_gpu
)

Loading YOLOv11 weights: yolo11n.pt
YOLOv11 model loaded on mps
imgsz set to 1920 for small player detection
DeepSORT tracker initialized for ultrawide frames:
  - max_age: 25
  - n_init: 3
  - max_iou_distance: 0.7
  - max_cosine_distance: 0.3
  - embedder: mobilenet


## Run Pipeline, Save Results

In [9]:
# filter to only test sequences
test_sequences = [s for s in sequences if 'test' in str(s['seq_dir'])]

print(f"\nProcessing {len(test_sequences)} test sequence(s) with YOLOv11 + DeepSORT...\n")

all_metrics = []

for seq in test_sequences:
    name = seq['name']
    video_path = seq['video_path']
    print(f"Processing: {name}")
    print(f"Video: {video_path}")

    metrics = process_sequence(
        detector=detector,
        tracker=tracker,
        video_path=video_path,
        seq_name=name,
        output_dir=RESULTS_ROOT,
        filter_classes=(PERSON_CLASS_ID,),
        save_video=True
    )

    if metrics:
        all_metrics.append(metrics)

# save metrics summary
metrics_file = RESULTS_ROOT / "metrics.json"

with open(metrics_file, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"\nTotal sequences processed: {len(all_metrics)}")
print(f"Results saved to: {RESULTS_ROOT}")
print(f"Metrics saved to: {metrics_file}")

if all_metrics:
    total_frames = sum(m['total_frames'] for m in all_metrics)
    total_time = sum(m['processing_time'] for m in all_metrics)
    avg_fps = sum(m['average_fps'] for m in all_metrics) / len(all_metrics)
    total_detections = sum(m['total_detections'] for m in all_metrics)

    print(f"\nSummary Statistics:")
    print(f"  - Total frames processed: {total_frames}")
    print(f"  - Total processing time: {total_time:.2f}s")
    print(f"  - Average FPS: {avg_fps:.2f}")
    print(f"  - Total detections: {total_detections}")


Processing 10 test sequence(s) with YOLOv11 + DeepSORT...

Processing: F_20220220_1_1890_1920
Video: /Users/echen/Documents/FA25/543/sports-data-tracker/data/soccer_side/test/F_20220220_1_1890_1920/img1.mp4


Processing F_20220220_1_1890_1920: 100%|██████████| 750/750 [11:52<00:00,  1.05it/s]


[F_20220220_1_1890_1920] Processed 750 frames in 712.18s (1.05 FPS)
[F_20220220_1_1890_1920] Total tracks written: 14707
[F_20220220_1_1890_1920] MOT output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/mot_outputs/F_20220220_1_1890_1920.txt
[F_20220220_1_1890_1920] Video output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/videos/F_20220220_1_1890_1920_tracked.mp4
Processing: F_20220220_1_1920_1950
Video: /Users/echen/Documents/FA25/543/sports-data-tracker/data/soccer_side/test/F_20220220_1_1920_1950/img1.mp4


Processing F_20220220_1_1920_1950: 100%|██████████| 750/750 [04:45<00:00,  2.63it/s]


[F_20220220_1_1920_1950] Processed 750 frames in 285.15s (2.63 FPS)
[F_20220220_1_1920_1950] Total tracks written: 4639
[F_20220220_1_1920_1950] MOT output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/mot_outputs/F_20220220_1_1920_1950.txt
[F_20220220_1_1920_1950] Video output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/videos/F_20220220_1_1920_1950_tracked.mp4
Processing: F_20220220_1_1680_1710
Video: /Users/echen/Documents/FA25/543/sports-data-tracker/data/soccer_side/test/F_20220220_1_1680_1710/img1.mp4


...

[F_20220220_1_1860_1890] Processed 750 frames in 392.39s (1.91 FPS)
[F_20220220_1_1860_1890] Total tracks written: 9221
[F_20220220_1_1860_1890] MOT output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/mot_outputs/F_20220220_1_1860_1890.txt
[F_20220220_1_1860_1890] Video output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/videos/F_20220220_1_1860_1890_tracked.mp4
Processing: F_20220220_1_1800_1830
Video: /Users/echen/Documents/FA25/543/sports-data-tracker/data/soccer_side/test/F_20220220_1_1800_1830/img1.mp4


Processing F_20220220_1_1800_1830: 100%|██████████| 750/750 [09:06<00:00,  1.37it/s]


[F_20220220_1_1800_1830] Processed 750 frames in 546.83s (1.37 FPS)
[F_20220220_1_1800_1830] Total tracks written: 12992
[F_20220220_1_1800_1830] MOT output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/mot_outputs/F_20220220_1_1800_1830.txt
[F_20220220_1_1800_1830] Video output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/videos/F_20220220_1_1800_1830_tracked.mp4
Processing: F_20220220_1_1710_1740
Video: /Users/echen/Documents/FA25/543/sports-data-tracker/data/soccer_side/test/F_20220220_1_1710_1740/img1.mp4


Processing F_20220220_1_1710_1740: 100%|██████████| 750/750 [04:55<00:00,  2.54it/s]

[F_20220220_1_1710_1740] Processed 750 frames in 295.15s (2.54 FPS)
[F_20220220_1_1710_1740] Total tracks written: 5424
[F_20220220_1_1710_1740] MOT output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/mot_outputs/F_20220220_1_1710_1740.txt
[F_20220220_1_1710_1740] Video output: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/videos/F_20220220_1_1710_1740_tracked.mp4

Total sequences processed: 10
Results saved to: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort
Metrics saved to: /Users/echen/Documents/FA25/543/sports-data-tracker/results/yolov11_deepsort/metrics.json

Summary Statistics:
  - Total frames processed: 7500
  - Total processing time: 4281.86s
  - Average FPS: 1.95
  - Total detections: 89875
